# Crypto Lab 07 — Diffie-Hellman Key Agreement

**CYBR 3570: Applied Cryptography**  
**Topic:** Public-key key agreement, discrete logarithms, protocol authentication, and KDFs

## Today's big question

> How can two people agree on a shared secret when an eavesdropper sees every message they exchange?

Diffie-Hellman is not an encryption algorithm by itself. It is a **key agreement** technique. Its output must still be authenticated, processed through a KDF, and used with symmetric cryptography.


## Learning objectives

By the end of this notebook, you should be able to:

1. Compute a toy Diffie-Hellman exchange using modular exponentiation.
2. Explain the relationship between DLP, CDH, and DDH at a high level.
3. Demonstrate why anonymous Diffie-Hellman fails against active attackers.
4. Explain the purpose of authenticated Diffie-Hellman.
5. Explain why raw DH shared secrets should be passed through a KDF.
6. Identify unsafe parameter choices and implementation risks.
7. Add an educational DH module to your cryptographic toolkit.


## Setup

In [1]:
import hashlib
import hmac
import secrets
from dataclasses import dataclass
from typing import Optional, Tuple, List

print("Lab 07 setup complete")

Lab 07 setup complete


## Part 1 — The Diffie-Hellman idea

Diffie-Hellman uses public values and private exponents.

Public parameters:

```text
p = a large prime
g = a generator/base
```

Alice chooses private exponent `a` and publishes:

```text
A = g^a mod p
```

Bob chooses private exponent `b` and publishes:

```text
B = g^b mod p
```

Alice computes:

```text
B^a mod p = (g^b)^a mod p = g^(ab) mod p
```

Bob computes:

```text
A^b mod p = (g^a)^b mod p = g^(ab) mod p
```

Both compute the same shared secret without ever sending `a`, `b`, or `g^(ab)` directly.


In [2]:
# A tiny toy Diffie-Hellman exchange.
# These parameters are intentionally small and insecure.

p = 23
g = 5

alice_private = 6
bob_private = 15

alice_public = pow(g, alice_private, p)
bob_public = pow(g, bob_private, p)

alice_shared = pow(bob_public, alice_private, p)
bob_shared = pow(alice_public, bob_private, p)

print(f"Public parameters: p={p}, g={g}")
print(f"Alice public value A = {alice_public}")
print(f"Bob public value   B = {bob_public}")
print(f"Alice shared secret = {alice_shared}")
print(f"Bob shared secret   = {bob_shared}")
print(f"Shared secrets match? {alice_shared == bob_shared}")

Public parameters: p=23, g=5
Alice public value A = 8
Bob public value   B = 19
Alice shared secret = 2
Bob shared secret   = 2
Shared secrets match? True


### Exercise 1 — Explain the algebra

In 3–5 sentences, explain why Alice and Bob compute the same shared value even though they use different private exponents.


### Answer:
The reason the shared math works even though they use the different A's and B's is because they have a link to the number via the shared p, g using the pow.
g^(ab) despite only knowing either a or b. Since A = g^a; A^b = (g^a)^b = g^(ab).
Both bob and alice compute the same values, but without the missing piece someone else could not compute the same shared secret.


## Part 2 — Generators and small groups

Diffie-Hellman works inside a mathematical group. In this toy notebook, we use the multiplicative group of nonzero integers modulo a prime `p`, often written `Z_p*`.

A generator is an element whose powers produce every nonzero group element.


In [3]:
def zn_star(p: int) -> list[int]:
    """Return the nonzero elements modulo p."""
    return list(range(1, p))


def powers_mod(g: int, p: int) -> list[int]:
    """Return g^1, g^2, ..., g^(p-1) modulo p."""
    return [pow(g, k, p) for k in range(1, p)]


def is_generator(g: int, p: int) -> bool:
    """Return True if g generates all of Z_p*. Assumes p is prime."""
    return set(powers_mod(g, p)) == set(zn_star(p))

p = 13
for g in zn_star(p):
    if is_generator(g, p):
        print(f"{g} is a generator of Z_{p}*: {powers_mod(g, p)}")

2 is a generator of Z_13*: [2, 4, 8, 3, 6, 12, 11, 9, 5, 10, 7, 1]
6 is a generator of Z_13*: [6, 10, 8, 9, 2, 12, 7, 3, 5, 4, 11, 1]
7 is a generator of Z_13*: [7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1]
11 is a generator of Z_13*: [11, 4, 5, 3, 7, 12, 2, 9, 8, 10, 6, 1]


### Exercise 2 — Find generators

Find all generators of `Z_17*`. For each generator, print its powers modulo 17.


In [4]:
# TODO: Find and print all generators of Z_17*.

p = 17
# Your code here
for g in zn_star(p):
    if is_generator(g, p):
        print(f"{g} is a generator of Z_{p}*: {powers_mod(g, p)}")


3 is a generator of Z_17*: [3, 9, 10, 13, 5, 15, 11, 16, 14, 8, 7, 4, 12, 2, 6, 1]
5 is a generator of Z_17*: [5, 8, 6, 13, 14, 2, 10, 16, 12, 9, 11, 4, 3, 15, 7, 1]
6 is a generator of Z_17*: [6, 2, 12, 4, 7, 8, 14, 16, 11, 15, 5, 13, 10, 9, 3, 1]
7 is a generator of Z_17*: [7, 15, 3, 4, 11, 9, 12, 16, 10, 2, 14, 13, 6, 8, 5, 1]
10 is a generator of Z_17*: [10, 15, 14, 4, 6, 9, 5, 16, 7, 2, 3, 13, 11, 8, 12, 1]
11 is a generator of Z_17*: [11, 2, 5, 4, 10, 8, 3, 16, 6, 15, 12, 13, 7, 9, 14, 1]
12 is a generator of Z_17*: [12, 8, 11, 13, 3, 2, 7, 16, 5, 9, 6, 4, 14, 15, 10, 1]
14 is a generator of Z_17*: [14, 9, 7, 13, 12, 15, 6, 16, 3, 8, 10, 4, 5, 2, 11, 1]


## Part 3 — Brute-forcing discrete logarithms

The discrete logarithm problem asks:

```text
Given g, x, and p, find a such that g^a = x mod p.
```

For tiny groups, brute force works. For cryptographic group sizes, brute force should be infeasible.


In [5]:
def discrete_log_bruteforce(g: int, x: int, p: int) -> Optional[int]:
    """Find a such that g^a = x mod p, or return None."""
    for a in range(0, p - 1):
        if pow(g, a, p) == x:
            return a
    return None

p = 23
g = 5
secret = 13
x = pow(g, secret, p)
recovered = discrete_log_bruteforce(g, x, p)

print(f"Given g={g}, p={p}, x={x}")
print(f"Recovered exponent: {recovered}")

Given g=5, p=23, x=21
Recovered exponent: 13


### Exercise 3 — Recover toy private exponents

Use `discrete_log_bruteforce()` to recover the private exponents for the following public values.


In [6]:
toy_public_values = [
    # (p, g, public_value)
    (23, 5, pow(5, 7, 23)),
    (23, 5, pow(5, 11, 23)),
    (29, 2, pow(2, 9, 29)),
]

# TODO: Recover the exponent for each public value.
for p, g, public_value in toy_public_values:
    recovered = discrete_log_bruteforce(g, public_value, p)
    # Your code here
    print(f"p={p}, g={g}, public={public_value}, recovered exponent={recovered}")

p=23, g=5, public=17, recovered exponent=7
p=23, g=5, public=22, recovered exponent=11
p=29, g=2, public=19, recovered exponent=9


## Part 4 — DLP, CDH, and DDH

Diffie-Hellman is connected to three related problems:

| Problem | Informal statement |
|---|---|
| DLP | Given `g^a`, recover `a`. |
| CDH | Given `g^a` and `g^b`, compute `g^(ab)`. |
| DDH | Given `g^a`, `g^b`, and `x`, decide whether `x = g^(ab)` or random. |

DLP is enough to break DH because recovering either private exponent lets an attacker compute the shared secret. CDH asks whether the attacker can compute the shared secret directly. DDH asks whether the attacker can distinguish the shared secret from a random group element.


### Exercise 4 — Concept check

For each scenario below, identify whether it most closely describes DLP, CDH, or DDH.


### Scenerios
- Recover Alice's private exponent from A = g^a mod p
    
    ***DLP***

- Compute g^(ab) after seeing g^a and g^b
    
    ***CDH***

- Decide whether a given value is the real shared secret or a random group element
    
    ***DDH***

## Part 5 — Anonymous Diffie-Hellman and man-in-the-middle attacks

Anonymous Diffie-Hellman gives Alice and Bob a shared secret against a passive eavesdropper. It does **not** authenticate either party.

An active attacker can sit between Alice and Bob and run two separate DH exchanges:

```text
Alice <---- shared secret 1 ----> Eve <---- shared secret 2 ----> Bob
```

Alice believes she shares a key with Bob. Bob believes he shares a key with Alice. In reality, both share different keys with Eve.


In [7]:
def dh_public(g: int, private: int, p: int) -> int:
    return pow(g, private, p)


def dh_shared(peer_public: int, private: int, p: int) -> int:
    return pow(peer_public, private, p)

p = 23
g = 5

# Alice and Bob's private exponents
alice_a = 6
bob_b = 15

# Eve chooses separate private exponents for each side
# Eve pretends to be Bob when talking to Alice, and Alice when talking to Bob.
eve_to_alice_d = 10
eve_to_bob_c = 4

# Public values
A = dh_public(g, alice_a, p)
B = dh_public(g, bob_b, p)
D = dh_public(g, eve_to_alice_d, p)  # sent to Alice instead of B
C = dh_public(g, eve_to_bob_c, p)    # sent to Bob instead of A

alice_thinks_shared = dh_shared(D, alice_a, p)
eve_with_alice = dh_shared(A, eve_to_alice_d, p)

bob_thinks_shared = dh_shared(C, bob_b, p)
eve_with_bob = dh_shared(B, eve_to_bob_c, p)

print("Alice's shared secret with Eve:", alice_thinks_shared, eve_with_alice)
print("Bob's shared secret with Eve:  ", bob_thinks_shared, eve_with_bob)
print("Alice and Bob share same secret?", alice_thinks_shared == bob_thinks_shared)

Alice's shared secret with Eve: 3 3
Bob's shared secret with Eve:   3 3
Alice and Bob share same secret? True


### Exercise 5 — Explain the MITM attack

Explain why the attack above works. Your answer should mention authentication.


In [9]:
exercise_5_answer = """
Your answer here.

 Since Eve can intercept and modify the public values exchanged between Alice and Bob, she can establish separate shared secrets with each of them.
 This allows her to decrypt and re-encrypt messages between Alice and Bob without their knowledge, effectively performing a man-in-the-middle attack. 
 As a result, Alice and Bob believe they are communicating securely with each other, while in reality, Eve has access to all their communications.

 Eve is able to intercept the public values A and B, and in the code they are replaced with D and C respectively. 
 This means that Alice computes her shared secret using D (which is derived from Eve's private key) instead of B,
 and Bob computes his shared secret using C (also derived from Eve's private key) instead of A.

"""
print(exercise_5_answer)


Your answer here.

 Since Eve can intercept and modify the public values exchanged between Alice and Bob, she can establish separate shared secrets with each of them.
 This allows her to decrypt and re-encrypt messages between Alice and Bob without their knowledge, effectively performing a man-in-the-middle attack. 
 As a result, Alice and Bob believe they are communicating securely with each other, while in reality, Eve has access to all their communications.

 Eve is able to intercept the public values A and B, and in the code they are replaced with D and C respectively. 
 This means that Alice computes her shared secret using D (which is derived from Eve's private key) instead of B,
 and Bob computes his shared secret using C (also derived from Eve's private key) instead of A.




## Part 6 — Authenticated Diffie-Hellman

Authenticated Diffie-Hellman adds identity verification. A common pattern is:

1. Alice sends her DH public value and a signature over that value.
2. Bob verifies the signature using Alice's known public signing key.
3. Bob sends his DH public value and a signature over that value.
4. Alice verifies Bob's signature.
5. Both compute the DH shared secret.

The important point is not the specific signature algorithm. The important point is that Eve cannot substitute her own DH public values unless she can forge signatures.


### Exercise 6 — Anonymous vs. authenticated DH

Complete the comparison table.


In [10]:
comparison = {
    "Anonymous DH protects against passive eavesdropping": "True",
    "Anonymous DH protects against active MITM": "False",
    "Authenticated DH uses long-term identity keys": "True",
    "Authenticated DH can provide forward secrecy when ephemeral DH exponents are erased": "True",
}

for statement, answer in comparison.items():
    print(statement)
    print("  ->", answer)

Anonymous DH protects against passive eavesdropping
  -> True
Anonymous DH protects against active MITM
  -> False
Authenticated DH uses long-term identity keys
  -> True
Authenticated DH can provide forward secrecy when ephemeral DH exponents are erased
  -> True


## Part 7 — Raw DH shared secrets are not keys

A DH shared secret is a group element, not a uniformly random byte string. Real protocols feed the DH shared secret into a KDF such as HKDF.

The KDF produces key material suitable for symmetric encryption and authentication.


In [11]:
def int_to_bytes(x: int) -> bytes:
    """Convert a positive integer to big-endian bytes."""
    if x == 0:
        return b"\x00"
    length = (x.bit_length() + 7) // 8
    return x.to_bytes(length, "big")


def hkdf_extract(salt: bytes, ikm: bytes, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Extract using HMAC."""
    return hmac.new(salt, ikm, hashmod).digest()


def hkdf_expand(prk: bytes, info: bytes, length: int, hashmod=hashlib.sha256) -> bytes:
    """HKDF-Expand for length bytes."""
    output = b""
    previous = b""
    counter = 1
    while len(output) < length:
        previous = hmac.new(prk, previous + info + bytes([counter]), hashmod).digest()
        output += previous
        counter += 1
    return output[:length]


def derive_session_key(shared_secret: int, context: bytes = b"CYBR3570-DH-demo") -> bytes:
    salt = b"CYBR3570 fixed demo salt"
    ikm = int_to_bytes(shared_secret)
    prk = hkdf_extract(salt, ikm)
    return hkdf_expand(prk, context, 32)

session_key = derive_session_key(alice_shared)
print("Raw shared secret:", alice_shared)
print("Derived session key:", session_key.hex())

Raw shared secret: 2
Derived session key: 0012340b89b78c112fc72ef942bb83e0f5f65b146d45c9ef834c83f4819749d3


### Exercise 7 — Derive separate keys

Use the same shared secret to derive two different 32-byte keys:

- one with context `b"encryption"`
- one with context `b"authentication"`

Explain why deriving separate keys for separate purposes is good protocol design.


In [12]:
# TODO: Derive two separate keys from alice_shared.

encryption_key = derive_session_key(alice_shared, b"encryption")
authentication_key = derive_session_key(alice_shared, b"authentication")

print("Encryption key:    ", encryption_key.hex() if encryption_key else None)
print("Authentication key:", authentication_key.hex() if authentication_key else None)

exercise_7_answer = """
Your explanation here.
Wanting to have two separate keys for encryption and authentication is a good practice in cryptography. 
It helps to ensure that the keys are used for their intended purposes and reduces the risk of key reuse,
 which can lead to vulnerabilities. By deriving separate keys from the shared secret, we can maintain the
   security of our communication and protect against potential attacks.
"""
print(exercise_7_answer)

Encryption key:     5e1be0f3e1861fe5375fbff8440f25e1050e980f9a7970a42179c9c1875cebde
Authentication key: 62429b2d87426fac1fc2a101b4b070648ef5cf785c2b45d9e8dbf09e377ad8f3

Your explanation here.
Wanting to have two separate keys for encryption and authentication is a good practice in cryptography. 
It helps to ensure that the keys are used for their intended purposes and reduces the risk of key reuse,
 which can lead to vulnerabilities. By deriving separate keys from the shared secret, we can maintain the
   security of our communication and protect against potential attacks.



## Part 8 — Unsafe parameters and small subgroups

Diffie-Hellman security depends on choosing safe parameters. Poor group choices can restrict possible shared secrets to a smaller set than expected or leak information about private exponents.

In real systems, do not invent your own group parameters. Use vetted standards and library defaults.


In [13]:
# Small example showing that a bad generator can produce a tiny set of values.

p = 23
for g in [1, 22, 2, 5]:
    values = sorted(set(powers_mod(g, p)))
    print(f"g={g:>2}: subgroup size={len(values):>2}, values={values}")

g= 1: subgroup size= 1, values=[1]
g=22: subgroup size= 2, values=[1, 22]
g= 2: subgroup size=11, values=[1, 2, 3, 4, 6, 8, 9, 12, 13, 16, 18]
g= 5: subgroup size=22, values=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


### Exercise 8 — Parameter review

For the values printed above, which choices of `g` obviously look unsafe? Why?


In [14]:
exercise_8_answer = """
Your answer here.
The g = 1 , g = 22, and g = 2 are not good generators because they do not produce all the elements of Z_23*.
- g = 1 produces only the value 1, which is a trivial subgroup of size 1.
- g = 22 produces only the values 22 and 1, which is a subgroup of size 2.
- g = 2 produces the values 2, 4, 8, 16, 9, 18, 13, 3, 6, 12, 1, which is a subgroup of size 11.
"""
print(exercise_8_answer)


Your answer here.
The g = 1 , g = 22, and g = 2 are not good generators because they do not produce all the elements of Z_23*.
- g = 1 produces only the value 1, which is a trivial subgroup of size 1.
- g = 22 produces only the values 22 and 1, which is a subgroup of size 2.
- g = 2 produces the values 2, 4, 8, 16, 9, 18, 13, 3, 6, 12, 1, which is a subgroup of size 11.



## Part 9 — Toolkit integration

Add the following module to your toolkit:

```text
crypto_toolkit/
    key_agreement/
        __init__.py
        diffie_hellman.py
```

Suggested functions:

```python
dh_public(g, private, p)
dh_shared(peer_public, private, p)
discrete_log_bruteforce(g, x, p)
derive_session_key(shared_secret, context)
is_generator(g, p)
```

Remember: this module is educational. Real projects should use established libraries and standardized groups.


## Engineering checkpoint

Answer the following questions in the dictionary below.


In [16]:
engineering_checkpoint = {
    "Why is Diffie-Hellman called key agreement rather than encryption?": "Encryption is different process than key agreement since key agreement is a protocol that lets two different groups compute a shared secret over an public channel without exposing the secret key. ",
    "Why does anonymous DH fail against active attackers?": "Active attackers are able to intercept and modify the public values between the two groups. This allows for the attacker to expose the communication from compromising the key exchange and shared secret.",
    "What does forward secrecy protect?": "Forward secrecy protects form previous keys from getting decrypted since they get generated every instance of in the key exchange. This ensures past communication isnt compromised and is protected.",
    "Why should protocols use a KDF after DH?": "KDF adds some randomness to the equation by adding hashing and extends the key. Fortifying and adding a non-uniform bit distribution and algebraic patterns allows for the KDF to be add to the DH.",
    "What should a security review check before accepting DH parameters?": "Checking the DH parameters is just a verification to ensure that the prime numbers are strong primes that are safe for use. The modulus length is important to be at least 2048 bits for security of the public key and as well review the generation of the key pairs and any weaknesses in the implementation of the DH protocol.",
}

for question, answer in engineering_checkpoint.items():
    print(question)
    print(answer)
    print()

Why is Diffie-Hellman called key agreement rather than encryption?
Encryption is different process than key agreement since key agreement is a protocol that lets two different groups compute a shared secret over an public channel without exposing the secret key. 

Why does anonymous DH fail against active attackers?
Active attackers are able to intercept and modify the public values between the two groups. This allows for the attacker to expose the communication from compromising the key exchange and shared secret.

What does forward secrecy protect?
Forward secrecy protects form previous keys from getting decrypted since they get generated every instance of in the key exchange. This ensures past communication isnt compromised and is protected.

Why should protocols use a KDF after DH?
KDF adds some randomness to the equation by adding hashing and extends the key. Fortifying and adding a non-uniform bit distribution and algebraic patterns allows for the KDF to be add to the DH.

What s

## Submission checklist

Before submitting, make sure you have:

- [ x] Completed all written responses.
- [ x] Implemented the missing code exercises.
- [ x] Run the notebook from top to bottom without errors.
- [ x] Added this week's functions to your cryptographic toolkit.
- [ x] Updated your engineering log.

## Final reflection

Return to today's big question:

> How can two people agree on a shared secret when an eavesdropper sees every message they exchange?

Explain the answer while also identifying what Diffie-Hellman does **not** solve by itself.


In [18]:
final_reflection = """
Your final reflection here.

Create good primes so that they are large enough to resist attacks,
and ensure that the generator used is a primitive root modulo the 
prime to maximize security. RSA has signatures and the key is the 
authenticate. Being able to understand the DH, forward secrecy, 
and the KDF is important to ensure that the key exchange is secure 
and that the shared secret is protected from potential attacks. 
It is also important to understand the limitations of anonymous 
DH and how it can be vulnerable to active attackers, and to use 
authenticated DH when possible to provide additional security.
DH inself is key agreement but it doesnt solve authentication problems. 
The encryption of the keys and KDF for the shared secret for the 
unpredicatable communication. Combining these techniques allows 
for two people to agree on a shared secret even when someone is 
eavesdropping.




"""
print(final_reflection)


Your final reflection here.

Create good primes so that they are large enough to resist attacks,
and ensure that the generator used is a primitive root modulo the 
prime to maximize security. RSA has signatures and the key is the 
authenticate. Being able to understand the DH, forward secrecy, 
and the KDF is important to ensure that the key exchange is secure 
and that the shared secret is protected from potential attacks. 
It is also important to understand the limitations of anonymous 
DH and how it can be vulnerable to active attackers, and to use 
authenticated DH when possible to provide additional security.
DH inself is key agreement but it doesnt solve authentication problems. 
The encryption of the keys and KDF for the shared secret for the 
unpredicatable communication. Combining these techniques allows 
for two people to agree on a shared secret even when someone is 
eavesdropping.





